# Solution: Week 2 Lab 2 — Exercise (Agentic email SDR)

Answers the **Exercise** at the end of [`2_lab2.ipynb`](../../2_lab2.ipynb) and extends the lab with **mail-merge / bulk send** tools.

**Environment:** run with the repo’s Python env (`openai-agents`, `sendgrid`). From this folder, `load_dotenv` loads `../../../.env` (repo root).

## 1. Agentic design patterns used in the lab

| Pattern | Where it appears |
|--------|-------------------|
| **Specialist agents** | Three sales personas (professional / engaging / busy). |
| **Parallelization** | `asyncio.gather` runs three `Runner.run` calls concurrently. |
| **Evaluator (LLM-as-judge)** | `sales_picker` picks the best email. |
| **Orchestrator / planner** | Sales Manager instructions: draft → evaluate → send (or hand off). |
| **Tool use** | `@function_tool` for `send_email` / `send_html_email`. |
| **Agent-as-tool** | `sales_agent1.as_tool(...)` — another agent exposed as a tool; control returns. |
| **Handoff** | `handoffs=[emailer_agent]` — control passes to Email Manager until done. |

Together: a small **multi-agent** system with **dynamic tool routing** and optional **delegation**.

## 2. “Workflow” vs “agent” (Anthropic) — the one line

In [Anthropic’s “Building effective agents”](https://www.anthropic.com/engineering/building-effective-agents), a **workflow** is mostly **fixed orchestration in code**. An **agent** is when the **LLM dynamically decides** how to use tools toward a goal.

The lab’s **workflow** phase uses explicit Python: `asyncio.gather`, string assembly, `Runner.run(sales_picker, emails)` — *you* define order.

Crossing into **agent** (model-driven orchestration) is giving the Sales Manager **tools** so the *model* chooses calls — concretely:

```python
sales_manager = Agent(..., tools=tools, ...)
```

So the practical “one line” is **`tools=tools`** on the Sales Manager `Agent`, plus `Runner.run(sales_manager, message)`.

## 3. Extension: mail merge — send to a list

- **`send_bulk_plaintext_emails`** — same body to many addresses (comma-separated).
- **`send_personalized_emails_json`** — JSON list of `{ "to": "...", "body": "..." }` for per-recipient bodies.

Use env vars for sender/recipients (see next cell).

In [ ]:
import json
import os
from pathlib import Path

import sendgrid
from dotenv import load_dotenv
from sendgrid.helpers.mail import Content, Email, Mail, To

from agents import Agent, Runner, function_tool, trace

_repo_env = Path("../../../.env").resolve()
if _repo_env.is_file():
    load_dotenv(_repo_env, override=True)
else:
    load_dotenv(override=True)

FROM_EMAIL = os.environ.get("SENDGRID_FROM_EMAIL", "you@verified-sender.example.com")
DEFAULT_RECIPIENTS = os.environ.get(
    "SENDGRID_RECIPIENT_LIST",
    "recipient1@example.com,recipient2@example.com",
)

In [ ]:
instructions1 = (
    "You are a sales agent working for ComplAI, a company that provides a SaaS tool for "
    "ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails."
)
instructions2 = (
    "You are a humorous, engaging sales agent working for ComplAI, a company that provides a SaaS tool for "
    "ensuring SOC2 compliance and preparing for audits, powered by AI. You write witty, engaging cold emails."
)
instructions3 = (
    "You are a busy sales agent working for ComplAI, a company that provides a SaaS tool for "
    "ensuring SOC2 compliance and preparing for audits, powered by AI. You write concise cold emails."
)

sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-4o-mini")
sales_agent2 = Agent(name="Engaging Sales Agent", instructions=instructions2, model="gpt-4o-mini")
sales_agent3 = Agent(name="Busy Sales Agent", instructions=instructions3, model="gpt-4o-mini")

In [ ]:
@function_tool
def send_email(body: str) -> dict:
    """Send one plaintext sales email to the default prospect (first address in list)."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    to = DEFAULT_RECIPIENTS.split(",")[0].strip()
    mail = Mail(Email(FROM_EMAIL), To(to), "Sales email", Content("text/plain", body)).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success", "to": to}


@function_tool
def send_bulk_plaintext_emails(body: str, recipient_emails: str) -> dict:
    """
    Send the same plaintext body to many recipients.
    recipient_emails: comma-separated addresses (e.g. "a@x.com,b@y.com").
    """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    emails = [e.strip() for e in recipient_emails.split(",") if e.strip()]
    sent = []
    for to_addr in emails:
        mail = Mail(Email(FROM_EMAIL), To(to_addr), "Sales email", Content("text/plain", body)).get()
        sg.client.mail.send.post(request_body=mail)
        sent.append(to_addr)
    return {"status": "success", "count": len(sent), "recipients": sent}


@function_tool
def send_personalized_emails_json(entries_json: str) -> dict:
    """
    Mail merge: different body per recipient.
    entries_json: JSON array of {"to": email, "body": plaintext}.
    """
    entries = json.loads(entries_json)
    if not isinstance(entries, list):
        raise ValueError("entries_json must be a JSON array")
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    sent = []
    for row in entries:
        to_addr = row["to"]
        body = row["body"]
        mail = Mail(Email(FROM_EMAIL), To(to_addr), "Sales email", Content("text/plain", body)).get()
        sg.client.mail.send.post(request_body=mail)
        sent.append(to_addr)
    return {"status": "success", "count": len(sent), "recipients": sent}

In [ ]:
desc = "Write a cold sales email"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=desc)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=desc)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=desc)

extended_tools = [tool1, tool2, tool3, send_email, send_bulk_plaintext_emails, send_personalized_emails_json]

manager_instructions = """
You are a Sales Manager at ComplAI. Produce the best cold outreach, then deliver it.

1. Use all three sales_agent tools to generate three drafts.
2. Pick the single best draft.
3. Sending:
   - For one default prospect, use send_email with that body.
   - For the SAME text to multiple people, use send_bulk_plaintext_emails with comma-separated recipient_emails.
   - For different bodies per person, use send_personalized_emails_json: JSON array of {"to", "body"}.

You must use sales_agent tools for drafts. One campaign per request (single send, one bulk, or one personalized batch).
"""

sales_manager_extended = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=extended_tools,
    model="gpt-4o-mini",
)

### Example runs

Requires `OPENAI_API_KEY`, `SENDGRID_API_KEY`, verified sender. Uncomment **one** block at a time.

In [ ]:
# with trace("Sales manager single"):
#     result = await Runner.run(
#         sales_manager_extended,
#         "Send a cold sales email addressed to 'Dear CEO'.",
#     )
#     print(result.final_output)

In [ ]:
# recipients = "alice@example.com,bob@example.com"
# with trace("Sales manager bulk"):
#     result = await Runner.run(
#         sales_manager_extended,
#         f"Send the same best cold email to these prospects: {recipients}",
#     )
#     print(result.final_output)

## 4. HARD challenge — replies via webhook (outline)

1. **Inbound Parse** (or similar) sends replies to your HTTPS URL.
2. **Verify** and parse payload (from, body, threading headers).
3. **Load** thread state from a store keyed by email / `Message-ID`.
4. **`Runner.run`** with history + new message; reply with `send_email` / HTML tools.

Use signed webhooks, idempotency, and avoid logging PII. For local dev, tunnel with **ngrok** and register the URL in SendGrid.